In [2]:

# ============================================================

# ============================================================
# STEP 1: Import Libraries
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully!")


# ============================================================
# STEP 2: Load Cleaned Dataset
# ============================================================

file_name = "Cleaned_Dataset_RFM.xlsx"

df = pd.read_excel(file_name)

print("\nDataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# ============================================================
# STEP 3: Display Dataset
# ============================================================

print("\n----- DATASET PREVIEW -----")
display(df.head())


# ============================================================
# STEP 4: Check Data Information
# ============================================================

print("\n----- DATA INFORMATION -----")
print(df.info())


# ============================================================
# STEP 5: Check Missing Values
# ============================================================

print("\n----- MISSING VALUES -----")
print(df.isnull().sum())


# ============================================================
# STEP 6: Convert Order_Date to Date Format
# ============================================================

df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

print("\nOrder_Date converted successfully.")


# ============================================================
# STEP 7: Basic Business Analysis
# ============================================================

total_customers = df["Customer_ID"].nunique()
total_orders = df["Order_ID"].nunique()
total_sales = df["Sales"].sum()
total_quantity = df["Quantity"].sum()
average_order_value = df["Sales"].mean()

print("\n----- BUSINESS SUMMARY -----")
print("Total Customers:", total_customers)
print("Total Orders:", total_orders)
print("Total Sales:", round(total_sales, 2))
print("Total Quantity:", total_quantity)
print("Average Order Value:", round(average_order_value, 2))


# ============================================================
# STEP 8: Set Analysis Date
# ============================================================

# The day after the latest order is used as the reference date.
analysis_date = df["Order_Date"].max() + pd.Timedelta(days=1)

print("\nAnalysis Date:", analysis_date.date())


# ============================================================
# STEP 9: Calculate RECENCY
# ============================================================

recency_df = (
    df.groupby("Customer_ID")["Order_Date"]
      .max()
      .reset_index()
)

recency_df["Recency"] = (
    analysis_date - recency_df["Order_Date"]
).dt.days

recency_df = recency_df[
    ["Customer_ID", "Recency"]
]

print("\n----- RECENCY -----")
display(recency_df.head())


# ============================================================
# STEP 10: Calculate FREQUENCY
# ============================================================

frequency_df = (
    df.groupby("Customer_ID")["Order_ID"]
      .nunique()
      .reset_index()
)

frequency_df.columns = [
    "Customer_ID",
    "Frequency"
]

print("\n----- FREQUENCY -----")
display(frequency_df.head())


# ============================================================
# STEP 11: Calculate MONETARY
# ============================================================

monetary_df = (
    df.groupby("Customer_ID")["Sales"]
      .sum()
      .reset_index()
)

monetary_df.columns = [
    "Customer_ID",
    "Monetary"
]

print("\n----- MONETARY -----")
display(monetary_df.head())


# ============================================================
# STEP 12: Create RFM Table
# ============================================================

rfm = (
    recency_df
    .merge(frequency_df, on="Customer_ID")
    .merge(monetary_df, on="Customer_ID")
)

print("\n----- RFM TABLE -----")
display(rfm.head())

print("\nRFM Table Shape:", rfm.shape)


# ============================================================
# STEP 13: Check RFM Statistics
# ============================================================

print("\n----- RFM STATISTICS -----")
display(
    rfm[
        ["Recency", "Frequency", "Monetary"]
    ].describe()
)


# ============================================================
# STEP 14: Create RFM Scores
# ============================================================

# Recency:
# Lower recency is better, therefore scoring is reversed.

rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)


# Frequency:
# Higher frequency is better.

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Monetary:
# Higher monetary value is better.

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


print("\n----- RFM SCORES -----")
display(
    rfm[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "R_Score",
            "F_Score",
            "M_Score"
        ]
    ].head(10)
)


# ============================================================
# STEP 15: Create RFM Score
# ============================================================

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

print("\nRFM Score created successfully.")


# ============================================================
# STEP 16: Create Total RFM Score
# ============================================================

rfm["RFM_Total_Score"] = (
    rfm["R_Score"]
    + rfm["F_Score"]
    + rfm["M_Score"]
)

print("\nRFM Total Score created successfully.")


# ============================================================
# STEP 17: Create Customer Segments
# ============================================================

def create_segment(row):

    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    # Best customers
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    # Loyal customers
    elif r >= 3 and f >= 4 and m >= 3:
        return "Loyal Customers"

    # High value but less recent
    elif r <= 2 and f >= 4 and m >= 4:
        return "At Risk High Value"

    # Customers who may be slipping
    elif r <= 2 and f >= 3:
        return "At Risk"

    # Recent customers
    elif r >= 4 and f <= 2:
        return "New / Promising"

    # Good monetary value
    elif m >= 4:
        return "Big Spenders"

    # Low engagement
    elif r <= 2 and f <= 2 and m <= 2:
        return "Lost Customers"

    else:
        return "Potential Customers"


rfm["Segment"] = rfm.apply(
    create_segment,
    axis=1
)

print("\n----- CUSTOMER SEGMENTS -----")
display(
    rfm[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "RFM_Score",
            "Segment"
        ]
    ].head(15)
)


# ============================================================
# STEP 18: Segment Size Analysis
# ============================================================

segment_size = (
    rfm.groupby("Segment")
       .size()
       .reset_index(name="Customer_Count")
)

segment_size["Percentage"] = (
    segment_size["Customer_Count"]
    / len(rfm) * 100
)

segment_size["Percentage"] = (
    segment_size["Percentage"].round(2)
)

segment_size = segment_size.sort_values(
    "Customer_Count",
    ascending=False
)

print("\n----- SEGMENT SIZE -----")
display(segment_size)


# ============================================================
# STEP 19: Segment Value Analysis
# ============================================================

segment_value = (
    rfm.groupby("Segment")
       .agg(
           Customer_Count=("Customer_ID", "nunique"),
           Total_Monetary=("Monetary", "sum"),
           Average_Monetary=("Monetary", "mean"),
           Average_Recency=("Recency", "mean"),
           Average_Frequency=("Frequency", "mean")
       )
       .reset_index()
)

segment_value["Total_Monetary"] = (
    segment_value["Total_Monetary"].round(2)
)

segment_value["Average_Monetary"] = (
    segment_value["Average_Monetary"].round(2)
)

segment_value["Average_Recency"] = (
    segment_value["Average_Recency"].round(2)
)

segment_value["Average_Frequency"] = (
    segment_value["Average_Frequency"].round(2)
)

segment_value = segment_value.sort_values(
    "Total_Monetary",
    ascending=False
)

print("\n----- SEGMENT VALUE ANALYSIS -----")
display(segment_value)


# ============================================================
# STEP 20: Customer Segment Action
# ============================================================

segment_actions = {
    "Champions":
        "Reward loyalty and provide premium offers.",

    "Loyal Customers":
        "Use loyalty programs and cross-selling.",

    "At Risk High Value":
        "Run personalized win-back campaigns.",

    "At Risk":
        "Send targeted discounts and re-engagement offers.",

    "New / Promising":
        "Encourage second purchase and onboarding.",

    "Big Spenders":
        "Offer premium products and exclusive benefits.",

    "Lost Customers":
        "Use win-back campaigns or limited-time offers.",

    "Potential Customers":
        "Nurture with personalized promotions."
}

segment_value["Recommended_Action"] = (
    segment_value["Segment"]
    .map(segment_actions)
)

print("\n----- SEGMENT ACTION PLAN -----")
display(segment_value)


# ============================================================
# STEP 21: Find Top Customers
# ============================================================

top_customers = (
    rfm.sort_values(
        ["Monetary", "Frequency"],
        ascending=[False, False]
    )
    .head(10)
)

print("\n----- TOP 10 CUSTOMERS -----")

display(
    top_customers[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "RFM_Score",
            "Segment"
        ]
    ]
)


# ============================================================
# STEP 22: Find Most Recent Customers
# ============================================================

recent_customers = (
    rfm.sort_values(
        "Recency",
        ascending=True
    )
    .head(10)
)

print("\n----- MOST RECENT CUSTOMERS -----")

display(
    recent_customers[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "RFM_Score",
            "Segment"
        ]
    ]
)


# ============================================================
# STEP 23: Find High Frequency Customers
# ============================================================

high_frequency = (
    rfm.sort_values(
        "Frequency",
        ascending=False
    )
    .head(10)
)

print("\n----- HIGH FREQUENCY CUSTOMERS -----")

display(
    high_frequency[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "RFM_Score",
            "Segment"
        ]
    ]
)


# ============================================================
# STEP 24: Overall RFM Summary
# ============================================================

print("\n============================================")
print("          RFM ANALYSIS SUMMARY")
print("============================================")

print("Total Customers:", len(rfm))

print(
    "Average Recency:",
    round(rfm["Recency"].mean(), 2)
)

print(
    "Average Frequency:",
    round(rfm["Frequency"].mean(), 2)
)

print(
    "Total Monetary Value:",
    round(rfm["Monetary"].sum(), 2)
)

print(
    "Average Monetary Value:",
    round(rfm["Monetary"].mean(), 2)
)

print(
    "Number of Segments:",
    rfm["Segment"].nunique()
)


# ============================================================
# STEP 25: Final RFM Dataset
# ============================================================

final_rfm = rfm[
    [
        "Customer_ID",
        "Recency",
        "Frequency",
        "Monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score",
        "RFM_Total_Score",
        "Segment"
    ]
].copy()

print("\n----- FINAL RFM DATASET -----")
display(final_rfm.head(10))


# ============================================================
# STEP 26: Create ONE Final Output File
# ============================================================

output_file = "RFM_Analysis_Final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # Sheet 1 - Customer RFM
    final_rfm.to_excel(
        writer,
        sheet_name="RFM Customers",
        index=False
    )

    # Sheet 2 - Segment Summary
    segment_value.to_excel(
        writer,
        sheet_name="Segment Summary",
        index=False
    )

    # Sheet 3 - Segment Size
    segment_size.to_excel(
        writer,
        sheet_name="Segment Size",
        index=False
    )

    # Sheet 4 - Top Customers
    top_customers.to_excel(
        writer,
        sheet_name="Top Customers",
        index=False
    )


# ============================================================
# STEP 27: Final Output
# ============================================================

print("\n============================================")
print("       RFM ANALYSIS COMPLETED")
print("============================================")

print("Final Output File:", output_file)
print("Total Customers:", len(final_rfm))
print("Total Segments:", final_rfm["Segment"].nunique())


# ============================================================
# STEP 28: Download ONE Output File
# ============================================================

from google.colab import files

files.download(output_file)

print("\nFinal RFM Analysis file downloaded successfully!")

Libraries imported successfully!

Dataset loaded successfully!
Rows: 983
Columns: 7

----- DATASET PREVIEW -----


,Customer_ID,Order_ID,Order_Date,Product_Category,Quantity,Unit_Price,Sales
0,CUST0103,ORD00001,2025-08-24,Furniture,4,9571.20,38284.80
1,CUST0180,ORD00002,2025-03-24,Electronics,3,11596.09,34788.27
2,CUST0015,ORD00004,2025-04-11,Office Supplies,1,12905.51,12905.51
3,CUST0107,ORD00005,2025-01-06,Electronics,4,2912.44,11649.76
4,CUST0072,ORD00006,2025-06-29,Furniture,5,8348.23,41741.15



----- DATA INFORMATION -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 983 entries, 0 to 982
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Customer_ID       983 non-null    object        
 1   Order_ID          983 non-null    object        
 2   Order_Date        983 non-null    datetime64[ns]
 3   Product_Category  983 non-null    object        
 4   Quantity          983 non-null    int64         
 5   Unit_Price        983 non-null    float64       
 6   Sales             983 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 53.9+ KB
None

----- MISSING VALUES -----
Customer_ID         0
Order_ID            0
Order_Date          0
Product_Category    0
Quantity            0
Unit_Price          0
Sales               0
dtype: int64

Order_Date converted successfully.

----- BUSINESS SUMMARY -----
Total Customers: 198
Total Order

,Customer_ID,Recency
0,CUST0001,10
1,CUST0002,6
2,CUST0003,84
3,CUST0004,178
4,CUST0005,57



----- FREQUENCY -----


...

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total_Score,Segment
0,CUST0001,10,8,169004.29,5,5,4,554,14,Champions
1,CUST0002,6,5,173875.90,5,3,4,534,12,Big Spenders
2,CUST0003,84,6,137635.48,2,4,4,244,10,At Risk High Value
3,CUST0004,178,4,151435.56,1,2,4,124,7,Big Spenders
4,CUST0005,57,6,136796.43,3,4,4,344,11,Loyal Customers
5,CUST0006,205,4,195092.71,1,2,5,125,8,Big Spenders
6,CUST0007,164,3,83207.73,1,1,2,112,4,Lost Customers
7,CUST0008,24,9,321061.36,4,5,5,455,14,Champions
8,CUST0009,53,5,74159.74,3,3,2,332,8,Potential Customers
9,CUST0010,280,2,43580.05,1,1,1,111,3,Lost Customers



       RFM ANALYSIS COMPLETED
Final Output File: RFM_Analysis_Final.xlsx
Total Customers: 198
Total Segments: 8


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Final RFM Analysis file downloaded successfully!
